# Load networks from an OSM PBF file

Author: [Geoff Boeing](https://geoffboeing.com/)

Load an OSM network from a local PBF file using a built-in network type or custom osmium filters.

  - [Documentation](https://osmnx.readthedocs.io/)
  - [Journal article and citation info](https://doi.org/10.1111/gean.70009)
  - [Code repository](https://github.com/gboeing/osmnx)
  - [Examples gallery](https://github.com/gboeing/osmnx-examples)

In [ ]:
# !uv pip install --system --quiet osmnx[all]
from pathlib import Path

import osmium
import osmnx as ox
import requests
from osmium import BackReferenceWriter as brw
from osmium import FileProcessor as fp

ox.settings.log_console = True
ox.__version__

## Download a PBF file

Download the District of Columbia extract from [Geofabrik](https://download.geofabrik.de/north-america/us/district-of-columbia.html), with data from [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). The remaining examples use this local file.

In [ ]:
# download the source file if it is not already cached
url = "https://download.geofabrik.de/north-america/us/district-of-columbia-latest.osm.pbf"
pbf_file = Path("./cache/dc.osm.pbf")
pbf_file.parent.mkdir(exist_ok=True)
if not pbf_file.is_file():
    pbf_file.write_bytes(requests.get(url, timeout=60).content)

## Use a built-in network type

In [ ]:
# model the bikeable network
G = ox.graph.graph_from_pbf(pbf_file, network_type="bike")

In [ ]:
# choose a subgraph around the white house, then plot
latlng = ox.geocoder.geocode("1600 Pennsylvania Avenue, Washington, DC")
node = ox.distance.nearest_nodes(G, latlng[1], latlng[0])
G = ox.truncate.truncate_graph_dist(G, source_node=node, dist=3000)
_, _ = ox.plot.plot_graph(G, node_size=0)

## Or, select cycleways with custom pre-filtering

Filter with osmium, then use `network_type=None` to load all selected ways and `retain_all=True` to keep disconnected components.

In [ ]:
# pre-filter pbf file to just its cycleways then build the graph
filtered_pbf = Path("./data/dc-cycleways.osm.pbf")
filtered_pbf.parent.mkdir(exist_ok=True)
tag_filter = osmium.filter.TagFilter(("highway", "cycleway"))
with brw(filtered_pbf, ref_src=pbf_file, remove_tags=False, overwrite=True) as writer:
    for way in fp(pbf_file, entities=osmium.osm.WAY).with_filter(tag_filter):
        writer.add_way(way)

G = ox.graph.graph_from_pbf(filtered_pbf, network_type=None, retain_all=True)

## Or, combine conditions with more complex bicycle pre-filtering

Select cycle streets, cycleways, living streets, designated bicycle paths, and roads with cycle tracks. These custom conditions select bicycle infrastructure independently of the built-in `bike` preset.

In [ ]:
# define the custom filter
def bicycle_filter(tags: dict[str, str]) -> bool:
    highway = tags.get("highway", "")
    if "cyclestreet" in tags or "cycleway" in highway or "living_street" in highway:
        return True
    if "path" in highway and "designated" in tags.get("bicycle", ""):
        return True
    keys = ("cycleway", "cycleway:right", "cycleway:left", "cycleway:both")
    return any("track" in tags.get(key, "") for key in keys)

In [ ]:
# apply the custom pre-filter and build the graph
filtered_pbf = Path("./data/dc-bicycle-network.osm.pbf")
with brw(filtered_pbf, ref_src=pbf_file, remove_tags=False, overwrite=True) as writer:
    for way in osmium.FileProcessor(pbf_file, entities=osmium.osm.WAY):
        if bicycle_filter(dict(way.tags)):
            writer.add_way(way)

G = ox.graph.graph_from_pbf(filtered_pbf, network_type=None, retain_all=True)

## Or, select all motorways

Select ways tagged `highway=motorway` and keep all disconnected components.

In [ ]:
# pre-filter pbf file to just its motorways
filtered_pbf = Path("./data/dc-motorways.osm.pbf")
tf = osmium.filter.TagFilter(("highway", "motorway"), ("highway", "motorway_link"))
with brw(filtered_pbf, ref_src=pbf_file, remove_tags=False, overwrite=True) as writer:
    for way in fp(pbf_file, entities=osmium.osm.WAY).with_filter(tf):
        writer.add_way(way)

In [ ]:
# build the graph then plot it
G = ox.graph.graph_from_pbf(filtered_pbf, network_type=None, retain_all=True)
_, _ = ox.plot.plot_graph(G, node_size=0)